# Laboratorio 24 — Mitigación de errores: ZNE y PEC

Este laboratorio implementa dos técnicas de mitigación de errores sin corrección de errores:

- **ZNE (Zero-Noise Extrapolation):** amplifica el ruido de forma controlada y extrapola a ruido cero.
- **PEC (Probabilistic Error Cancellation):** cancela el ruido representando el canal ideal como combinación cuasi-probabilística de canales ruidosos.

**Prerrequisitos:** Módulos 16 (canales cuánticos), 28 (benchmarking NISQ).

**Qiskit:** 2.0+ con `qiskit-aer`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

print("Imports OK")

## 1. Circuito de referencia

Usamos un circuito sencillo que prepara un estado conocido. El valor ideal de ⟨Z⟩ es 1 (estado |0⟩).
Aplicamos una secuencia de puertas que equivale a identidad: X·X = I.

In [ ]:
def make_circuit(n_x_pairs: int = 5) -> QuantumCircuit:
    """Circuito con 2*n_x_pairs puertas X (equivalente a identidad)."""
    qc = QuantumCircuit(1)
    for _ in range(n_x_pairs):
        qc.x(0)
        qc.x(0)
    return qc

qc_base = make_circuit(n_x_pairs=5)
print(qc_base.draw("text"))

# Valor ideal
sv = Statevector.from_instruction(qc_base)
Z = SparsePauliOp("Z")
print(f"\n⟨Z⟩ ideal = {sv.expectation_value(Z).real:.4f}")

## 2. Simulación con ruido base

Añadimos ruido despolarizante con tasa `p` a cada puerta X.

In [ ]:
def build_noise_model(p_error: float) -> NoiseModel:
    nm = NoiseModel()
    error = depolarizing_error(p_error, 1)
    nm.add_all_qubit_quantum_error(error, ["x"])
    return nm

def measure_z_noisy(qc: QuantumCircuit, p_error: float, shots: int = 8192) -> float:
    """Estima ⟨Z⟩ con ruido usando AerSimulator."""
    qc_m = qc.copy()
    qc_m.measure_all()
    nm = build_noise_model(p_error)
    sim = AerSimulator()
    job = sim.run(qc_m, noise_model=nm, shots=shots)
    counts = job.result().get_counts()
    total = sum(counts.values())
    # ⟨Z⟩ = P(0) - P(1)
    p0 = counts.get("0", 0) / total
    p1 = counts.get("1", 0) / total
    return p0 - p1

p_base = 0.02   # 2% de error por puerta
shots  = 8192
qc_base = make_circuit(n_x_pairs=5)

z_ideal  = 1.0
z_noisy  = measure_z_noisy(qc_base, p_base, shots)

print(f"⟨Z⟩ ideal:   {z_ideal:.4f}")
print(f"⟨Z⟩ ruidoso: {z_noisy:.4f}  (degradación: {z_ideal - z_noisy:.4f})")

## 3. ZNE — Zero-Noise Extrapolation

### 3.1 Gate folding

Para amplificar el ruido por un factor λ, reemplazamos cada puerta U por U(U†U)^{(λ-1)/2}.
Con λ = 1, 3, 5 se consiguen factores de escala enteros.

Para puertas X (que son auto-inversas, X† = X):
- λ=1: X
- λ=3: X·X·X = X  (pero con 3 veces el ruido)
- λ=5: X·X·X·X·X = X (5 veces el ruido)

In [ ]:
def fold_circuit(qc: QuantumCircuit, scale_factor: int) -> QuantumCircuit:
    """Gate folding: repite cada puerta (2k+1) veces para scale_factor = 2k+1."""
    assert scale_factor % 2 == 1, "scale_factor debe ser impar"
    k = (scale_factor - 1) // 2
    qc_folded = QuantumCircuit(qc.num_qubits)
    for instruction in qc.data:
        # Aplicar la puerta original + k veces (puerta inversa + puerta)
        qc_folded.append(instruction.operation, instruction.qubits)
        for _ in range(k):
            qc_folded.append(instruction.operation.inverse(), instruction.qubits)
            qc_folded.append(instruction.operation, instruction.qubits)
    return qc_folded

# Verificar que el circuito plegado da el mismo resultado ideal
lambdas = [1, 3, 5]
for lam in lambdas:
    qc_f = fold_circuit(qc_base, lam)
    sv_f = Statevector.from_instruction(qc_f)
    print(f"λ={lam}: ⟨Z⟩ ideal con folding = {sv_f.expectation_value(Z).real:.4f}, n_puertas={qc_f.size()}")

### 3.2 Medición a distintos factores de escala

In [ ]:
lambdas = [1, 3, 5]
z_values = []

for lam in lambdas:
    qc_f = fold_circuit(qc_base, lam)
    z = measure_z_noisy(qc_f, p_base, shots)
    z_values.append(z)
    print(f"λ={lam}: ⟨Z⟩ = {z:.4f}")

### 3.3 Extrapolación lineal y de Richardson

In [ ]:
lambdas_arr = np.array(lambdas, dtype=float)
z_arr = np.array(z_values)

# Extrapolación lineal (regresión de grado 1)
coeffs_lin = np.polyfit(lambdas_arr, z_arr, 1)
z0_lineal = np.polyval(coeffs_lin, 0)

# Extrapolación polinómica de grado 2 (Richardson)
coeffs_poly = np.polyfit(lambdas_arr, z_arr, 2)
z0_poly = np.polyval(coeffs_poly, 0)

print(f"⟨Z⟩ ideal (referencia):     {z_ideal:.4f}")
print(f"⟨Z⟩ ruidoso (λ=1):          {z_values[0]:.4f}")
print(f"⟨Z⟩ ZNE lineal (λ→0):       {z0_lineal:.4f}")
print(f"⟨Z⟩ ZNE Richardson (λ→0):   {z0_poly:.4f}")

print(f"\nError sin mitigación: {abs(z_ideal - z_values[0]):.4f}")
print(f"Error ZNE lineal:      {abs(z_ideal - z0_lineal):.4f}")
print(f"Error ZNE Richardson:  {abs(z_ideal - z0_poly):.4f}")

### 3.4 Curva de extrapolación

In [ ]:
lam_plot = np.linspace(0, 6, 100)

fig, ax = plt.subplots(figsize=(9, 5))

# Datos medidos
ax.scatter(lambdas_arr, z_arr, color="#3498db", s=100, zorder=5, label="Medido")

# Extrapolación lineal
ax.plot(lam_plot, np.polyval(coeffs_lin, lam_plot), "--", color="#e74c3c",
        label=f"Lineal → {z0_lineal:.4f}")

# Extrapolación Richardson (polinomio grado 2)
ax.plot(lam_plot, np.polyval(coeffs_poly, lam_plot), "-", color="#2ecc71",
        label=f"Richardson → {z0_poly:.4f}")

# Valor ideal
ax.axhline(y=z_ideal, color="black", linestyle=":", alpha=0.7, label=f"Ideal = {z_ideal:.4f}")
ax.axvline(x=0, color="gray", linestyle=":", alpha=0.3)

ax.scatter([0], [z0_lineal], color="#e74c3c", s=120, zorder=6, marker="D")
ax.scatter([0], [z0_poly], color="#2ecc71", s=120, zorder=6, marker="D")

ax.set_xlabel("Factor de escala λ")
ax.set_ylabel("⟨Z⟩")
ax.set_title("ZNE — Curva de extrapolación a ruido cero")
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(-0.3, 6)
plt.tight_layout()
plt.show()

## 4. Comparativa: ideal / ruidoso / ZNE mitigado

Barremos distintas tasas de error base y comparamos los tres escenarios.

In [ ]:
p_vals = [0.005, 0.01, 0.02, 0.03, 0.05]
lambdas = [1, 3, 5]

z_noisy_list  = []
z_zne_lin_list = []
z_zne_pol_list = []

for p in p_vals:
    z_meas = []
    for lam in lambdas:
        qc_f = fold_circuit(qc_base, lam)
        z_m = measure_z_noisy(qc_f, p, shots)
        z_meas.append(z_m)
    
    z_noisy_list.append(z_meas[0])
    
    la = np.array(lambdas, dtype=float)
    za = np.array(z_meas)
    z_zne_lin_list.append(np.polyval(np.polyfit(la, za, 1), 0))
    z_zne_pol_list.append(np.polyval(np.polyfit(la, za, 2), 0))
    print(f"p={p:.3f}: ruidoso={z_meas[0]:.4f}, ZNE_lin={z_zne_lin_list[-1]:.4f}, ZNE_rich={z_zne_pol_list[-1]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.axhline(y=1.0, color="black", linestyle=":", alpha=0.7, label="Ideal = 1.0")
ax.plot(p_vals, z_noisy_list, "o-", color="#e74c3c", linewidth=2, label="Ruidoso (sin mitigación)")
ax.plot(p_vals, z_zne_lin_list, "s--", color="#3498db", linewidth=2, label="ZNE lineal")
ax.plot(p_vals, z_zne_pol_list, "^-", color="#2ecc71", linewidth=2, label="ZNE Richardson")

ax.set_xlabel("Tasa de error física p")
ax.set_ylabel("⟨Z⟩ estimado")
ax.set_title("Comparativa: sin mitigación vs. ZNE")
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0.5, 1.05)
plt.tight_layout()
plt.show()

## 5. Métricas: fidelidad recuperada y overhead de shots

In [ ]:
# Para p_base = 0.02 (el caso principal)
p = 0.02
z_meas_base = []
for lam in [1, 3, 5]:
    qc_f = fold_circuit(qc_base, lam)
    z_meas_base.append(measure_z_noisy(qc_f, p, shots))

la = np.array([1., 3., 5.])
za = np.array(z_meas_base)

z0_noisy = za[0]
z0_zne_lin = np.polyval(np.polyfit(la, za, 1), 0)
z0_zne_rich = np.polyval(np.polyfit(la, za, 2), 0)

def fidelidad_recuperada(z_est, z_ideal=1.0, z_noisy=z0_noisy):
    """Fraccion del error original recuperada."""
    error_total = abs(z_ideal - z_noisy)
    error_mitigado = abs(z_ideal - z_est)
    if error_total == 0:
        return 1.0
    return 1 - error_mitigado / error_total

print("=== Métricas de mitigación ===")
print(f"Valor ideal:          1.0000")
print(f"Valor ruidoso:        {z0_noisy:.4f}")
print(f"ZNE lineal:           {z0_zne_lin:.4f}   | Fidelidad recuperada: {fidelidad_recuperada(z0_zne_lin)*100:.1f}%")
print(f"ZNE Richardson:       {z0_zne_rich:.4f}   | Fidelidad recuperada: {fidelidad_recuperada(z0_zne_rich)*100:.1f}%")
print()
print("=== Overhead de shots ===")
print(f"Sin mitigación:       {shots:,} shots  (1× overhead)")
print(f"ZNE con 3 factores:   {shots*3:,} shots (3× overhead)")
print(f"Overhead por shot para mismo error estadístico: {3:.0f}×")

## 6. Resumen y conclusiones

| Método | Ventaja | Limitación |
|---|---|---|
| Sin mitigación | Sin overhead | Error sistemático proporcional a p·n_gates |
| ZNE lineal | 2-3× reducción de error, bajo overhead | Asume modelo de ruido lineal |
| ZNE Richardson | Mayor reducción, válido para ruidos más complejos | 3-5× overhead de shots |
| PEC | Cancelación exacta del error | Overhead exponencial: 2^(p·n_gates) shots |

**ZNE** es la técnica práctica más usada en hardware NISQ actual:
- Implementada en Qiskit IBM Runtime como `ZNEStrategy`.
- Overhead manejable: 3-5× shots para circuitos moderados.
- Mejora real medida en experimentos de química cuántica y QAOA.

**PEC** es teóricamente exacto pero su overhead exponencial lo hace impractical para circuitos con más de 20-30 puertas ruidosas.

**Módulos relacionados:** 16 (canales cuánticos), 28 (benchmarking), 29 (fault-tolerance).